<a href="https://colab.research.google.com/github/TASRUNNER2010/Hackathon/blob/main/Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import pandas as pd
from pathlib import Path

# Locate the folder containing all four original EPA files.
folder = Path(
    "/content/drive/MyDrive/Colab Notebooks/veritas AI project"
)

# List only the original files so we don't accidentally load cleaned outputs.
filenames = [
    "2023_ad_viz_plotval_data.csv",
    "2024_ad_viz_plotval_data.csv",
    "2025_ad_viz_plotval_data.csv",
    "2026_ad_viz_plotval_data.csv"
]

# Read each file and preserve identifiers as text.
tables = []

for filename in filenames:
    table = pd.read_csv(
        folder / filename,
        dtype={"Site ID": "string", "POC": "string"}
    )

    # Remove surrounding spaces from column names.
    table.columns = table.columns.str.strip()

    # Record which file each measurement came from.
    table["source_file"] = filename
    tables.append(table)

# Combine the years and preserve an independent raw copy.
raw_df = pd.concat(tables, ignore_index=True)

# Select the Rutgers station and its mostly daily POC 3 series.
df = raw_df.loc[
    raw_df["Site ID"].str.strip().eq("340230011")
    & raw_df["POC"].str.strip().eq("3")
].copy()

# Stop if the expected station and instrument were not found.
if df.empty:
    raise ValueError("No records found for Rutgers POC 3.")

# Keep measurements and metadata useful for checking data quality.
df = df[[
    "Date",
    "Daily Mean PM2.5 Concentration",
    "Daily AQI Value",
    "Source",
    "Method Code",
    "Method Description",
    "source_file"
]].rename(columns={
    "Date": "date",
    "Daily Mean PM2.5 Concentration": "pm25",
    "Daily AQI Value": "aqi",
    "Source": "source",
    "Method Code": "method_code",
    "Method Description": "method_description"
})

# Convert date strings into dates and measurements into numbers.
df["date"] = pd.to_datetime(
    df["date"], format="%m/%d/%Y", errors="raise"
)

for column in ["pm25", "aqi"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

# Remove repeated records, ignoring the filename when comparing rows.
record_columns = [c for c in df.columns if c != "source_file"]
df = df.drop_duplicates(subset=record_columns)

# Investigate conflicting same-day records rather than choosing arbitrarily.
if df["date"].duplicated().any():
    raise ValueError("Multiple different records exist for the same date.")

# Build a complete calendar from the actual selected date range.
# This replaces the old calendar that was restricted to 2023.
calendar = pd.date_range(
    start=df["date"].min(),
    end=df["date"].max(),
    name="date"
)

# Insert missing dates without inventing pollution measurements.
df = (
    df.set_index("date")
      .sort_index()
      .reindex(calendar)
      .reset_index()
)

# Flag missing pollution measurements and preliminary AirNow records.
df["measurement_missing"] = df["pm25"].isna()
df["is_airnow"] = df["source"].eq("AirNow")

# Summarize measurement coverage separately for each year.
summary = df.groupby(df["date"].dt.year).agg(
    calendar_days=("date", "size"),
    measured_days=("pm25", "count"),
    missing_days=("measurement_missing", "sum"),
    airnow_records=("is_airnow", "sum")
)

display(summary)
display(df.head())

# Save a new combined file without overwriting any original CSV.
df.to_csv(folder / "airaware_clean_daily_2023_2026.csv", index=False)

,calendar_days,measured_days,missing_days,airnow_records
date,,,,
2023,365,337,28,0
2024,366,348,18,0
2025,365,354,11,0
2026,317,292,25,130


,date,pm25,aqi,source,method_code,method_description,source_file,measurement_missing,is_airnow
0,2023-01-01,5.6,31.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False
1,2023-01-02,9.9,52.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False
2,2023-01-03,15.3,63.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False
3,2023-01-04,12.3,57.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False
4,2023-01-05,12.5,57.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False


In [12]:
import requests

# Use the first and last dates in the cleaned EPA dataset.
# strftime converts dates into the format Open-Meteo expects.
start_date = df["date"].min().strftime("%Y-%m-%d")
end_date = df["date"].max().strftime("%Y-%m-%d")

# Open-Meteo's historical weather API.
url = "https://archive-api.open-meteo.com/v1/archive"

# Specify the location, date range, weather variables, and units.
params = {
    # Coordinates of the Rutgers EPA monitoring station.
    "latitude": 40.462182,
    "longitude": -74.429439,

    # Match the full date range of the cleaned EPA data.
    "start_date": start_date,
    "end_date": end_date,

    # Daily average temperature, humidity, and wind speed,
    # plus total daily precipitation.
    "daily": ",".join([
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_mean"
    ]),

    # Match weather days to local New Jersey calendar days.
    "timezone": "America/New_York",

    # Request explicit measurement units.
    "temperature_unit": "celsius",
    "precipitation_unit": "mm",
    "wind_speed_unit": "ms"
}

# Download the weather data.
# Stop if Open-Meteo returns an HTTP error.
response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

# Convert the response from JSON into a daily weather table.
weather_data = response.json()
weather = pd.DataFrame(weather_data["daily"])

# Shorten column names and include units in their names.
weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature_c",
    "relative_humidity_2m_mean": "humidity_pct",
    "precipitation_sum": "precipitation_mm",
    "wind_speed_10m_mean": "wind_speed_ms"
})

# Convert weather date strings into pandas dates.
weather["date"] = pd.to_datetime(weather["date"])

# Ensure weather values are numeric.
weather_columns = [
    "temperature_c",
    "humidity_pct",
    "precipitation_mm",
    "wind_speed_ms"
]

for column in weather_columns:
    weather[column] = pd.to_numeric(weather[column], errors="raise")

# Match weather and pollution records by date.
# Preserve every EPA calendar day, including missing measurements.
# Stop if either table has duplicate dates.
combined = df.merge(
    weather,
    on="date",
    how="left",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

# Report the requested period and missing weather values.
print(f"Requested weather: {start_date} to {end_date}")
print("Missing weather values:")
print(combined[weather_columns].isna().sum())

# Show a sample of the merged pollution-and-weather dataset.
display(combined.head(10))

# Save weather separately so it can be reused without another download.
weather.to_csv(
    folder / "airaware_weather_2023_2026.csv",
    index=False
)

# Save the complete merged dataset without overwriting the EPA-only file.
combined.to_csv(
    folder / "airaware_epa_weather_2023_2026.csv",
    index=False
)

Requested weather: 2023-01-01 to 2026-11-13
Missing weather values:
temperature_c       0
humidity_pct        0
precipitation_mm    0
wind_speed_ms       0
dtype: int64


,date,pm25,aqi,source,method_code,method_description,source_file,measurement_missing,is_airnow,temperature_c,humidity_pct,precipitation_mm,wind_speed_ms
0,2023-01-01,5.6,31.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False,9.5,83,0.8,3.24
1,2023-01-02,9.9,52.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False,9.2,83,0.8,1.67
2,2023-01-03,15.3,63.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False,11.2,94,6.2,2.36
3,2023-01-04,12.3,57.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False,15.0,91,2.1,2.64
4,2023-01-05,12.5,57.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False,11.3,92,0.0,2.41
5,2023-01-06,NaN,NaN,NaN,NaN,NaN,NaN,True,False,6.6,84,6.9,2.93
6,2023-01-07,NaN,NaN,NaN,NaN,NaN,NaN,True,False,3.8,68,0.0,3.59
7,2023-01-08,NaN,NaN,NaN,NaN,NaN,NaN,True,False,1.6,71,0.0,1.97
8,2023-01-09,NaN,NaN,NaN,NaN,NaN,NaN,True,False,3.5,75,1.7,2.83
9,2023-01-10,13.5,59.0,AQS,183.0,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,2023_ad_viz_plotval_data.csv,False,False,2.8,68,0.0,2.58


In [13]:
# Work on a separate copy of the complete daily calendar.
forecast_data = combined.sort_values("date").copy()

# Use the following calendar day's PM2.5 as the prediction target.
forecast_data["pm25_tomorrow"] = forecast_data["pm25"].shift(-1)

# Record the date being predicted so we can split by target date.
forecast_data["target_date"] = (
    forecast_data["date"] + pd.Timedelta(days=1)
)

# Start with today's pollution and today's weather as inputs.
features = [
    "pm25",
    "temperature_c",
    "humidity_pct",
    "precipitation_mm",
    "wind_speed_ms"
]

# Keep only examples with complete inputs and a real next-day measurement.
# Missing pollution is never replaced with zero or an invented target.
model_df = forecast_data.dropna(
    subset=features + ["pm25_tomorrow"]
).copy()

# Split chronologically using the date being predicted.
train_df = model_df[
    model_df["target_date"] < "2025-01-01"
].copy()

validation_df = model_df[
    (model_df["target_date"] >= "2025-01-01")
    & (model_df["target_date"] < "2026-01-01")
].copy()

test_df = model_df[
    model_df["target_date"] >= "2026-01-01"
].copy()

# Check how many forecasting examples are available in each period.
print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Test examples:", len(test_df))

# Inspect the input date, target date, and actual next-day pollution.
display(model_df[
    ["date", "target_date", "pm25", "pm25_tomorrow"]
].head())

Training examples: 674
Validation examples: 350
Test examples: 285


,date,target_date,pm25,pm25_tomorrow
0,2023-01-01,2023-01-02,5.6,9.9
1,2023-01-02,2023-01-03,9.9,15.3
2,2023-01-03,2023-01-04,15.3,12.3
3,2023-01-04,2023-01-05,12.3,12.5
9,2023-01-10,2023-01-11,13.5,6.0


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Select the training inputs and the next-day measurements to learn from.
X_train = train_df[features]
y_train = train_df["pm25_tomorrow"]

# Create a Linear Regression model.
model = LinearRegression()

# Train the model using only the training period.
model.fit(X_train, y_train)

# Predict next-day pollution for the validation examples.
validation_predictions = model.predict(validation_df[features])

# Baseline: tomorrow's pollution equals today's pollution.
baseline_predictions = validation_df["pm25"]

# Get the actual next-day measurements for those same examples.
actual = validation_df["pm25_tomorrow"]

# Compare both approaches using mean absolute error.
baseline_mae = mean_absolute_error(actual, baseline_predictions)
model_mae = mean_absolute_error(actual, validation_predictions)

# Lower MAE means smaller prediction errors on average.
print(f"Validation baseline MAE: {baseline_mae:.2f} µg/m³")
print(f"Validation Linear Regression MAE: {model_mae:.2f} µg/m³")

Validation baseline MAE: 2.60 µg/m³
Validation Linear Regression MAE: 2.41 µg/m³
